# Generate an Answer from Retrieved Evidence

In [ ]:
# Run before importing the libraries.
%pip install -q "opentelemetry-api>=1.39.0,<=1.42.1" "opentelemetry-sdk>=1.39.0,<=1.42.1" "chromadb>=1.0,<2" "sentence-transformers>=3,<6" "transformers>=4.45,<5" "accelerate>=0.26,<2"

In [ ]:
import chromadb
import sentence_transformers
import transformers
import accelerate

print("ChromaDB:", chromadb.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)
print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)

print("\nAll RAG packages imported successfully!")

ChromaDB: 1.5.9
Sentence Transformers: 5.7.0
Transformers: 4.57.6
Accelerate: 1.14.0

All RAG packages imported successfully!


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

BASE_DIR = "/content/drive/MyDrive/PhishDetectAI"
CHROMA_PATH = os.path.join(BASE_DIR, "rag_chroma_correct")

print("Project path:", BASE_DIR)
print("ChromaDB path:", CHROMA_PATH)
print("Project path exists:", os.path.exists(BASE_DIR))
print("ChromaDB path exists:", os.path.exists(CHROMA_PATH))

Project path: /content/drive/MyDrive/PhishDetectAI
ChromaDB path: /content/drive/MyDrive/PhishDetectAI/rag_chroma_correct
Project path exists: True
ChromaDB path exists: True


# Create ChromaDB

In [ ]:
import chromadb

client = chromadb.PersistentClient(path=CHROMA_PATH)

collection = client.get_or_create_collection(
    name="phishing_emails"
)

print("New ChromaDB collection created successfully.")
print("Collection name:", collection.name)
print("Documents currently stored:", collection.count())

New ChromaDB collection created successfully.
Collection name: phishing_emails
Documents currently stored: 23813


#load the prepared training data

In [ ]:
import json
import os

TRAIN_PATH = os.path.join(
    BASE_DIR,
    "datasets",
    "prepared_data",
    "train.jsonl"
)

with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    train_data = [json.loads(line) for line in f]

print("Training examples loaded:", len(train_data))
print("First example:")
print(train_data[0])

Training examples loaded: 23813
First example:
{'instruction': 'Classify the following email as phishing or legitimate.', 'input': "Subject: barrister thambo\n\nBody: i am barrister andrew thambo an attorney of clifford alexander ' s attorney in johannesburg , south africa . a special client of mine mandated me to look for over sea ' s company , agent or an individual who can act as his over sea ' s partner ' s in his proposed investment in real estate / property in their country and also investment in some blue chip company ' s\r\nhowever the companies or individual ' s involvement will be properly paid for although the term ' s and condition ' s are negotiable ?\r\nexpecting your swift response . our second chamber ' s email is andy _ thambo _ jhb @ yahoo . com\r\nregard ' s\r\nbarrister andrew thambo", 'output': 'phishing'}


# Check Label Distribution

In [ ]:
from collections import Counter

labels = [item["output"] for item in train_data]

label_counts = Counter(labels)

print("Label distribution:")
for label, count in label_counts.items():
    print(f"{label}: {count}")

Label distribution:
phishing: 11173
legitimate: 12640


# Verify all required fields/Check structure

In [ ]:
required_fields = {"instruction", "input", "output"}

missing_fields = []

for i, item in enumerate(train_data):
    missing = required_fields - set(item.keys())

    if missing:
        missing_fields.append((i, missing))

print("Total examples:", len(train_data))
print("Examples with missing fields:", len(missing_fields))

if missing_fields:
    print("First problem:", missing_fields[0])
else:
    print("All examples have the required fields.")

Total examples: 23813
Examples with missing fields: 0
All examples have the required fields.


# Load Embedding Model

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully.")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Embedding model loaded successfully.


In [ ]:
import chromadb

client = chromadb.PersistentClient(
    path=CHROMA_PATH
)

collection = client.get_or_create_collection(
    name="phishing_emails"
)

print("Collection:", collection.name)
print("Document count:", collection.count())

Collection: phishing_emails
Document count: 23813


# Prepare documents. metadata and IDs

In [ ]:
documents = [
    item["input"]
    for item in train_data
]

metadatas = [
    {
        "label": item["output"]
    }
    for item in train_data
]

ids = [
    f"email_train_{i}"
    for i in range(len(train_data))
]

print("Documents:", len(documents))
print("Metadata:", len(metadatas))
print("IDs:", len(ids))

print("\nFirst ID:", ids[0])
print("First label:", metadatas[0]["label"])

Documents: 23813
Metadata: 23813
IDs: 23813

First ID: email_train_0
First label: phishing


# Generate embeddings in batches

In [ ]:
import numpy as np

BATCH_SIZE = 64

all_embeddings = []

for start in range(0, len(documents), BATCH_SIZE):
    batch = documents[start:start + BATCH_SIZE]

    batch_embeddings = embedding_model.encode(
        batch,
        batch_size=BATCH_SIZE,
        show_progress_bar=False
    )

    all_embeddings.extend(batch_embeddings.tolist())

    if start % (BATCH_SIZE * 10) == 0:
        print(f"Processed {min(start + BATCH_SIZE, len(documents))}/{len(documents)}")

print("\nEmbedding generation complete.")
print("Number of embeddings:", len(all_embeddings))
print("Embedding dimensions:", len(all_embeddings[0]))

Processed 64/23813
Processed 704/23813
Processed 1344/23813
Processed 1984/23813
Processed 2624/23813
Processed 3264/23813
Processed 3904/23813
Processed 4544/23813
Processed 5184/23813
Processed 5824/23813
Processed 6464/23813
Processed 7104/23813
Processed 7744/23813
Processed 8384/23813
Processed 9024/23813
Processed 9664/23813
Processed 10304/23813
Processed 10944/23813
Processed 11584/23813
Processed 12224/23813
Processed 12864/23813
Processed 13504/23813
Processed 14144/23813
Processed 14784/23813
Processed 15424/23813
Processed 16064/23813
Processed 16704/23813
Processed 17344/23813
Processed 17984/23813
Processed 18624/23813
Processed 19264/23813
Processed 19904/23813
Processed 20544/23813
Processed 21184/23813
Processed 21824/23813
Processed 22464/23813
Processed 23104/23813
Processed 23744/23813

Embedding generation complete.
Number of embeddings: 23813
Embedding dimensions: 384


# Save the embeddings

In [ ]:
import numpy as np
import os

EMBEDDINGS_DIR = os.path.join(
    BASE_DIR,
    "embeddings"
)

os.makedirs(EMBEDDINGS_DIR, exist_ok=True)

EMBEDDINGS_PATH = os.path.join(
    EMBEDDINGS_DIR,
    "email_embeddings_correct.npy"
)

np.save(
    EMBEDDINGS_PATH,
    np.array(all_embeddings, dtype=np.float32)
)

print("Embeddings saved successfully!")
print("Path:", EMBEDDINGS_PATH)
print("Number of embeddings:", len(all_embeddings))
print("Dimensions:", len(all_embeddings[0]))

Embeddings saved successfully!
Path: /content/drive/MyDrive/PhishDetectAI/embeddings/email_embeddings_correct.npy
Number of embeddings: 23813
Dimensions: 384


### Verify the saved embeddings

In [ ]:
all_embeddings = np.load(
    EMBEDDINGS_PATH
).tolist()

print("Embeddings loaded:", len(all_embeddings))
print("Dimensions:", len(all_embeddings[0]))

Embeddings loaded: 23813
Dimensions: 384


# Add existing all_embeddings to ChromaDB

In [ ]:
BATCH_SIZE = 500

for start in range(0, len(documents), BATCH_SIZE):

    end = min(start + BATCH_SIZE, len(documents))

    collection.upsert(
        ids=ids[start:end],
        documents=documents[start:end],
        metadatas=metadatas[start:end],
        embeddings=all_embeddings[start:end]
    )

    print(f"Added {end}/{len(documents)} documents")

Added 500/23813 documents
Added 1000/23813 documents
Added 1500/23813 documents
Added 2000/23813 documents
Added 2500/23813 documents
Added 3000/23813 documents
Added 3500/23813 documents
Added 4000/23813 documents
Added 4500/23813 documents
Added 5000/23813 documents
Added 5500/23813 documents
Added 6000/23813 documents
Added 6500/23813 documents
Added 7000/23813 documents
Added 7500/23813 documents
Added 8000/23813 documents
Added 8500/23813 documents
Added 9000/23813 documents
Added 9500/23813 documents
Added 10000/23813 documents
Added 10500/23813 documents
Added 11000/23813 documents
Added 11500/23813 documents
Added 12000/23813 documents
Added 12500/23813 documents
Added 13000/23813 documents
Added 13500/23813 documents
Added 14000/23813 documents
Added 14500/23813 documents
Added 15000/23813 documents
Added 15500/23813 documents
Added 16000/23813 documents
Added 16500/23813 documents
Added 17000/23813 documents
Added 17500/23813 documents
Added 18000/23813 documents
Added 18500/

### Verify the collection

In [ ]:
print("ChromaDB document count:", collection.count())

# Check one stored document
check = collection.get(
    ids=["email_train_0"],
    include=["documents", "metadatas"]
)

print("\nStored document:")
print(check["documents"][0])

print("\nStored metadata:")
print(check["metadatas"][0])

ChromaDB document count: 23813

Stored document:
Subject: barrister thambo

Body: i am barrister andrew thambo an attorney of clifford alexander ' s attorney in johannesburg , south africa . a special client of mine mandated me to look for over sea ' s company , agent or an individual who can act as his over sea ' s partner ' s in his proposed investment in real estate / property in their country and also investment in some blue chip company ' s
however the companies or individual ' s involvement will be properly paid for although the term ' s and condition ' s are negotiable ?
expecting your swift response . our second chamber ' s email is andy _ thambo _ jhb @ yahoo . com
regard ' s
barrister andrew thambo

Stored metadata:
{'label': 'phishing'}


### Test RAG retrieval

In [ ]:
test_email = """
Subject: Urgent account suspension

Body: Your bank account will be suspended today.
Please verify your account immediately by clicking the link below.
"""

# Create embedding for the test email
query_embedding = embedding_model.encode(test_email).tolist()

# Search ChromaDB for the 3 most similar emails
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3,
    include=["documents", "metadatas", "distances"]
)

print("Top 3 retrieved emails:\n")

for i in range(3):
    print("=" * 70)
    print(f"Result {i+1}")
    print("Label:", results["metadatas"][0][i]["label"])
    print("Distance:", results["distances"][0][i])
    print("Email:")
    print(results["documents"][0][i][:500])

Top 3 retrieved emails:

Result 1
Label: phishing
Distance: 0.6536012887954712
Email:
Subject: citibank alerting service

Body: dear citibank customer we were unable to
process the recent transactions on
your account . to ensure that your account is not
suspended , please update your information by clicking
here .
if you have recently updated
your information , please
disregard this message as we are processing the
changes you have made .
citibank customer service
copyright  2004 citicorp
- - >

Result 2
Label: phishing
Distance: 0.6995491981506348
Email:
Subject: note ! citibank account suspend in process

Body: dear customer :
recently there have been a large number of cyber attacks pointing our database servers . in order to safeguard your account , we require you to sign on immediately .
this personal check is requested of you as a precautionary measure and to ensure yourselves that everything is normal with your balance and personal information .
this process is mandatory , and if

### Load the test dataset

In [ ]:
import json

TEST_PATH = os.path.join(
    BASE_DIR,
    "datasets",
    "prepared_data",
    "test.jsonl"
)

with open(TEST_PATH, "r", encoding="utf-8") as f:
    test_data = [json.loads(line) for line in f]

print("Test examples:", len(test_data))
print("First example:")
print(test_data[0])

Test examples: 5954
First example:
{'instruction': 'Classify the following email as phishing or legitimate.', 'input': 'Subject: re : credit risk technical article\n\nBody: ben ,\r\nlet me review it one more time from this angle . if you don \' t hear from me by\r\nwednesday ,\r\nit \' s ok to post it .\r\nvince\r\nbenjamin parsons\r\n01 / 04 / 2000 04 : 31 am\r\nto : vince j kaminski / hou / ect @ ect\r\ncc : ross prevatt / hou / ect @ ect , bijoya banerjea / lon / ect @ ect\r\nsubject : credit risk technical article\r\nvince ,\r\nbijoya is currently working on the development of the website for the new\r\ncredit trading initiative , and we thought it would be a good idea to put my\r\nrecent technical article " measuring credit risk " on it . are you okay with\r\nthis release , or does it contain any proprietary information we should edit\r\nout ?\r\nben', 'output': 'legitimate'}


### Prepare Test emails and labels

In [ ]:
test_documents = [item["input"] for item in test_data]
test_labels = [item["output"] for item in test_data]

print("Test documents:", len(test_documents))
print("Test labels:", len(test_labels))

print("\nLabel distribution:")
from collections import Counter
print(Counter(test_labels))

print("\nFirst test label:", test_labels[0])

Test documents: 5954
Test labels: 5954

Label distribution:
Counter({'legitimate': 3151, 'phishing': 2803})

First test label: legitimate


### Evaluate RAG retrieval

In [ ]:
import numpy as np

# Evaluate on the first 1000 test emails
EVAL_SIZE = 1000

top1_correct = 0
top3_correct = 0

for i in range(EVAL_SIZE):

    # Create embedding for the test email
    query_embedding = embedding_model.encode(
        test_documents[i],
        show_progress_bar=False
    ).tolist()

    # Retrieve top 3 similar emails
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=3,
        include=["metadatas"]
    )

    retrieved_labels = [
        metadata["label"]
        for metadata in results["metadatas"][0]
    ]

    actual_label = test_labels[i]

    # Top-1
    if retrieved_labels[0] == actual_label:
        top1_correct += 1

    # Top-3
    if actual_label in retrieved_labels:
        top3_correct += 1

top1_recall = top1_correct / EVAL_SIZE
top3_recall = top3_correct / EVAL_SIZE

print("RAG Retrieval Evaluation")
print("=" * 40)
print(f"Evaluation emails: {EVAL_SIZE}")
print(f"Top-1 correct: {top1_correct}")
print(f"Top-1 Recall: {top1_recall:.4f} ({top1_recall * 100:.2f}%)")
print()
print(f"Top-3 correct: {top3_correct}")
print(f"Top-3 Recall: {top3_recall:.4f} ({top3_recall * 100:.2f}%)")

RAG Retrieval Evaluation
Evaluation emails: 1000
Top-1 correct: 982
Top-1 Recall: 0.9820 (98.20%)

Top-3 correct: 998
Top-3 Recall: 0.9980 (99.80%)


### Save Retrieval results

In [ ]:
retrieval_results = {
    "evaluation_size": EVAL_SIZE,
    "top_1_correct": top1_correct,
    "top_1_recall": top1_recall,
    "top_3_correct": top3_correct,
    "top_3_recall": top3_recall
}

print("Retrieval results saved:")
print(retrieval_results)

Retrieval results saved:
{'evaluation_size': 1000, 'top_1_correct': 982, 'top_1_recall': 0.982, 'top_3_correct': 998, 'top_3_recall': 0.998}


# Save results to drive

In [ ]:
import json
import os

RESULTS_DIR = os.path.join(BASE_DIR, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

RETRIEVAL_RESULTS_PATH = os.path.join(
    RESULTS_DIR,
    "rag_retrieval_results_correct.json"
)

with open(RETRIEVAL_RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(retrieval_results, f, indent=4)

print("Retrieval results saved successfully!")
print("Path:", RETRIEVAL_RESULTS_PATH)

Retrieval results saved successfully!
Path: /content/drive/MyDrive/PhishDetectAI/results/rag_retrieval_results_correct.json


# Create/ Prepare TF-IDF baseline

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create TF-IDF vectorizer
tfidf_vectorizer = TfidfVectorizer(
    max_features=50000,
    stop_words="english"
)

# Fit TF-IDF on the training emails
train_tfidf = tfidf_vectorizer.fit_transform(documents)

print("TF-IDF training matrix created successfully.")
print("Number of documents:", train_tfidf.shape[0])
print("Number of features:", train_tfidf.shape[1])

TF-IDF training matrix created successfully.
Number of documents: 23813
Number of features: 50000


# Evaluate TF-IDF retrieval

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

tfidf_top1_correct = 0
tfidf_top3_correct = 0

for i in range(EVAL_SIZE):

    # Convert test email into TF-IDF representation
    test_vector = tfidf_vectorizer.transform([test_documents[i]])

    # Calculate similarity with all training emails
    similarities = cosine_similarity(
        test_vector,
        train_tfidf
    )[0]

    # Get indices of the 3 most similar emails
    top_indices = np.argsort(similarities)[-3:][::-1]

    retrieved_labels = [
        metadatas[index]["label"]
        for index in top_indices
    ]

    actual_label = test_labels[i]

    # Top-1
    if retrieved_labels[0] == actual_label:
        tfidf_top1_correct += 1

    # Top-3
    if actual_label in retrieved_labels:
        tfidf_top3_correct += 1

tfidf_top1_recall = tfidf_top1_correct / EVAL_SIZE
tfidf_top3_recall = tfidf_top3_correct / EVAL_SIZE

print("TF-IDF Retrieval Evaluation")
print("=" * 40)
print(f"Evaluation emails: {EVAL_SIZE}")
print(f"Top-1 correct: {tfidf_top1_correct}")
print(f"Top-1 Recall: {tfidf_top1_recall:.4f} ({tfidf_top1_recall * 100:.2f}%)")
print()
print(f"Top-3 correct: {tfidf_top3_correct}")
print(f"Top-3 Recall: {tfidf_top3_recall:.4f} ({tfidf_top3_recall * 100:.2f}%)")

TF-IDF Retrieval Evaluation
Evaluation emails: 1000
Top-1 correct: 970
Top-1 Recall: 0.9700 (97.00%)

Top-3 correct: 995
Top-3 Recall: 0.9950 (99.50%)


# Save TF-IDF results

In [ ]:
tfidf_results = {
    "evaluation_size": EVAL_SIZE,
    "top_1_correct": tfidf_top1_correct,
    "top_1_recall": tfidf_top1_recall,
    "top_3_correct": tfidf_top3_correct,
    "top_3_recall": tfidf_top3_recall
}

TFIDF_RESULTS_PATH = os.path.join(
    RESULTS_DIR,
    "tfidf_retrieval_results_correct.json"
)

with open(TFIDF_RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(tfidf_results, f, indent=4)

print("TF-IDF results saved successfully!")
print("Path:", TFIDF_RESULTS_PATH)

TF-IDF results saved successfully!
Path: /content/drive/MyDrive/PhishDetectAI/results/tfidf_retrieval_results_correct.json


# Load Fine-Tunned Qwen

In [ ]:
!pip install unsloth

# Restore projects path

In [ ]:
import os

BASE_DIR = "/content/drive/MyDrive/PhishDetectAI"

CHROMA_PATH = os.path.join(BASE_DIR, "rag_chroma_correct")
EMBEDDINGS_PATH = os.path.join(
    BASE_DIR,
    "embeddings",
    "email_embeddings_correct.npy"
)
RESULTS_DIR = os.path.join(BASE_DIR, "results")
QWEN_MODEL_PATH = os.path.join(
    BASE_DIR,
    "models",
    "qwen_phishing_unsloth_correct"
)

print("Project exists:", os.path.exists(BASE_DIR))
print("ChromaDB exists:", os.path.exists(CHROMA_PATH))
print("Embeddings exist:", os.path.exists(EMBEDDINGS_PATH))
print("Results folder exists:", os.path.exists(RESULTS_DIR))
print("Qwen model exists:", os.path.exists(QWEN_MODEL_PATH))

Project exists: True
ChromaDB exists: True
Embeddings exist: True
Results folder exists: True
Qwen model exists: True


### Load Fine-tunned Qwen model

In [ ]:
import unsloth

from unsloth import FastLanguageModel

qwen_model, qwen_tokenizer = FastLanguageModel.from_pretrained(
    model_name=QWEN_MODEL_PATH,
    max_seq_length=2048,
    load_in_4bit=True
)

print("Fine-tuned Qwen model loaded successfully!")
print("Model path:", QWEN_MODEL_PATH)

==((====))==  Unsloth 2026.9.11: Fast Qwen2 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Fine-tuned Qwen model loaded successfully!
Model path: /content/drive/MyDrive/PhishDetectAI/models/qwen_phishing_unsloth_correct


# Confirm the model and tokenizer

In [ ]:
print("Qwen model loaded:", qwen_model is not None)
print("Qwen tokenizer loaded:", qwen_tokenizer is not None)
print("GPU available:", qwen_model.device)

Qwen model loaded: True
Qwen tokenizer loaded: True
GPU available: cuda:0


# Create the RAG prediction function

In [ ]:
def predict_rag(email_text):

    # --------------------------------------------------
    # 1. Create embedding for the email
    # --------------------------------------------------

    query_embedding = embedding_model.encode(
        email_text,
        show_progress_bar=False
    ).tolist()


    # --------------------------------------------------
    # 2. Retrieve top 3 similar emails
    # --------------------------------------------------

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=3,
        include=["documents", "metadatas", "distances"]
    )


    # --------------------------------------------------
    # 3. Build RAG evidence
    # --------------------------------------------------

    evidence_text = ""

    for i in range(3):

        retrieved_email = results["documents"][0][i][:300]
        retrieved_label = results["metadatas"][0][i]["label"]

        evidence_text += (
            f"\n--- Evidence {i + 1} ---\n"
            f"Label: {retrieved_label}\n"
            f"Email: {retrieved_email}\n"
        )


    # --------------------------------------------------
    # 4. Use the same system prompt as fine-tuned Qwen
    # --------------------------------------------------

    SYSTEM_PROMPT = (
        "You are a cybersecurity expert specializing in phishing detection. "
        "Analyze the provided email, message, or URL. "
        "Determine whether it is a phishing attempt or a legitimate communication. "
        "Respond with only one word: 'phishing' or 'legitimate'."
    )


    # --------------------------------------------------
    # 5. Create Qwen chat messages
    # --------------------------------------------------

    rag_user_content = f"""
Classify the following email.

Email:
{email_text[:1500]}

Similar email examples retrieved from the database:
{evidence_text}

Use the similar examples as supporting evidence.
Classify the ORIGINAL email.

Respond with only one word:
phishing
or
legitimate
"""

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": rag_user_content
        }
    ]


    # --------------------------------------------------
    # 6. Apply Qwen chat template
    # --------------------------------------------------

    formatted_input = qwen_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )


    # --------------------------------------------------
    # 7. Tokenize
    # --------------------------------------------------

    inputs = qwen_tokenizer(
        formatted_input,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(qwen_model.device)


    # --------------------------------------------------
    # 8. Generate prediction
    # --------------------------------------------------

    outputs = qwen_model.generate(
        **inputs,
        max_new_tokens=5,
        do_sample=False,
        eos_token_id=qwen_tokenizer.eos_token_id
    )


    # --------------------------------------------------
    # 9. Decode only newly generated tokens
    # --------------------------------------------------

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    raw_output = qwen_tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip().lower()


    # --------------------------------------------------
    # 10. Extract classification
    # --------------------------------------------------

    if "phishing" in raw_output:
        return "phishing"

    elif "legitimate" in raw_output:
        return "legitimate"

    else:
        return "invalid"

# Test RAG prediction (one example)

In [ ]:
test_email = """
Subject: Urgent account suspension

Body: Your bank account will be suspended today.
Please verify your account immediately by clicking the link below.
"""

prediction = predict_rag(test_email)

print("Test email prediction:", prediction)

Test email prediction: phishing


In [ ]:
prediction = predict_rag(test_email)

print("Final RAG prediction:", prediction)

Final RAG prediction: phishing


# Reload the test dataset and prepare test_documents

In [ ]:
import json
from collections import Counter

TEST_PATH = os.path.join(
    BASE_DIR,
    "datasets",
    "prepared_data",
    "test.jsonl"
)

with open(TEST_PATH, "r", encoding="utf-8") as f:
    test_data = [json.loads(line) for line in f]

test_documents = [item["input"] for item in test_data]
test_labels = [item["output"] for item in test_data]

print("Test examples loaded:", len(test_data))
print("Test documents:", len(test_documents))
print("Test labels:", len(test_labels))
print("Label distribution:", Counter(test_labels))

Test examples loaded: 5954
Test documents: 5954
Test labels: 5954
Label distribution: Counter({'legitimate': 3151, 'phishing': 2803})


# Test RAG + Qwen on 10 emails first

In [ ]:
import time

TEST_SIZE = 10
rag_predictions = []
rag_actual_labels = []

start_time = time.time()

for i in range(TEST_SIZE):

    prediction = predict_rag(test_documents[i])

    rag_predictions.append(prediction)
    rag_actual_labels.append(test_labels[i])

    print(
        f"{i + 1}/{TEST_SIZE} | "
        f"Actual: {test_labels[i]} | "
        f"Predicted: {prediction}"
    )

elapsed_time = time.time() - start_time

print("\n" + "=" * 50)
print("RAG + Qwen test completed")
print("Emails tested:", TEST_SIZE)
print("Time taken:", round(elapsed_time, 2), "seconds")

1/10 | Actual: legitimate | Predicted: legitimate
2/10 | Actual: phishing | Predicted: phishing
3/10 | Actual: phishing | Predicted: invalid
4/10 | Actual: phishing | Predicted: phishing
5/10 | Actual: legitimate | Predicted: legitimate
6/10 | Actual: phishing | Predicted: phishing
7/10 | Actual: legitimate | Predicted: legitimate
8/10 | Actual: legitimate | Predicted: legitimate
9/10 | Actual: legitimate | Predicted: legitimate
10/10 | Actual: phishing | Predicted: phishing

RAG + Qwen test completed
Emails tested: 10
Time taken: 8.32 seconds


# Calculate the metrics for the 10-email test.

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# --------------------------------------------------
# Remove invalid predictions
# --------------------------------------------------

valid_indices = [
    i for i, prediction in enumerate(rag_predictions)
    if prediction in ["legitimate", "phishing"]
]

valid_actual = [
    rag_actual_labels[i]
    for i in valid_indices
]

valid_predictions = [
    rag_predictions[i]
    for i in valid_indices
]

invalid_count = len(rag_predictions) - len(valid_predictions)

# --------------------------------------------------
# Calculate metrics
# --------------------------------------------------

accuracy = accuracy_score(
    valid_actual,
    valid_predictions
)

precision = precision_score(
    valid_actual,
    valid_predictions,
    pos_label="phishing"
)

recall = recall_score(
    valid_actual,
    valid_predictions,
    pos_label="phishing"
)

f1 = f1_score(
    valid_actual,
    valid_predictions,
    pos_label="phishing"
)

cm = confusion_matrix(
    valid_actual,
    valid_predictions,
    labels=["legitimate", "phishing"]
)

# --------------------------------------------------
# Display results
# --------------------------------------------------

print("=" * 60)
print("RAG + Qwen FINAL EVALUATION")
print("=" * 60)

print(f"Total emails       : {len(rag_predictions)}")
print(f"Valid predictions  : {len(valid_predictions)}")
print(f"Invalid predictions: {invalid_count}")

print()
print(f"Accuracy : {accuracy:.4f} ({accuracy * 100:.2f}%)")
print(f"Precision: {precision:.4f} ({precision * 100:.2f}%)")
print(f"Recall   : {recall:.4f} ({recall * 100:.2f}%)")
print(f"F1 Score : {f1:.4f} ({f1 * 100:.2f}%)")

print("\nConfusion Matrix")
print("                 Predicted")
print("                 Legitimate  Phishing")
print(f"Actual Legitimate    {cm[0][0]:3d}        {cm[0][1]:3d}")
print(f"Actual Phishing      {cm[1][0]:3d}        {cm[1][1]:3d}")

print("\nClassification Report")
print(
    classification_report(
        valid_actual,
        valid_predictions,
        labels=["legitimate", "phishing"],
        target_names=["legitimate", "phishing"]
    )
)

RAG + Qwen FINAL EVALUATION
Total emails       : 10
Valid predictions  : 9
Invalid predictions: 1

Accuracy : 1.0000 (100.00%)
Precision: 1.0000 (100.00%)
Recall   : 1.0000 (100.00%)
F1 Score : 1.0000 (100.00%)

Confusion Matrix
                 Predicted
                 Legitimate  Phishing
Actual Legitimate      5          0
Actual Phishing        0          4

Classification Report
              precision    recall  f1-score   support

  legitimate       1.00      1.00      1.00         5
    phishing       1.00      1.00      1.00         4

    accuracy                           1.00         9
   macro avg       1.00      1.00      1.00         9
weighted avg       1.00      1.00      1.00         9



# Test 100 emails

In [ ]:
import time

TEST_SIZE = 100

rag_predictions = []
rag_actual_labels = []

start_time = time.time()

for i in range(TEST_SIZE):

    prediction = predict_rag(test_documents[i])

    rag_predictions.append(prediction)
    rag_actual_labels.append(test_labels[i])

    print(
        f"{i + 1}/{TEST_SIZE} | "
        f"Actual: {test_labels[i]} | "
        f"Predicted: {prediction}"
    )

elapsed_time = time.time() - start_time

print("\n" + "=" * 50)
print("RAG + Qwen test completed")
print("Emails tested:", TEST_SIZE)
print("Time taken:", round(elapsed_time, 2), "seconds")

1/100 | Actual: legitimate | Predicted: legitimate
2/100 | Actual: phishing | Predicted: phishing
3/100 | Actual: phishing | Predicted: invalid
4/100 | Actual: phishing | Predicted: phishing
5/100 | Actual: legitimate | Predicted: legitimate
6/100 | Actual: phishing | Predicted: phishing
7/100 | Actual: legitimate | Predicted: legitimate
8/100 | Actual: legitimate | Predicted: legitimate
9/100 | Actual: legitimate | Predicted: legitimate
10/100 | Actual: phishing | Predicted: phishing
11/100 | Actual: phishing | Predicted: phishing
12/100 | Actual: legitimate | Predicted: legitimate
13/100 | Actual: legitimate | Predicted: legitimate
14/100 | Actual: legitimate | Predicted: legitimate
15/100 | Actual: legitimate | Predicted: legitimate
16/100 | Actual: legitimate | Predicted: legitimate
17/100 | Actual: legitimate | Predicted: legitimate
18/100 | Actual: legitimate | Predicted: legitimate
19/100 | Actual: legitimate | Predicted: legitimate
20/100 | Actual: legitimate | Predicted: legit

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Keep only valid predictions
valid_indices = [
    i for i, prediction in enumerate(rag_predictions)
    if prediction in ["legitimate", "phishing"]
]

valid_predictions = [
    rag_predictions[i]
    for i in valid_indices
]

valid_actual_labels = [
    rag_actual_labels[i]
    for i in valid_indices
]

invalid_count = len(rag_predictions) - len(valid_predictions)

accuracy = accuracy_score(
    valid_actual_labels,
    valid_predictions
)

precision = precision_score(
    valid_actual_labels,
    valid_predictions,
    pos_label="phishing",
    zero_division=0
)

recall = recall_score(
    valid_actual_labels,
    valid_predictions,
    pos_label="phishing",
    zero_division=0
)

f1 = f1_score(
    valid_actual_labels,
    valid_predictions,
    pos_label="phishing",
    zero_division=0
)

cm = confusion_matrix(
    valid_actual_labels,
    valid_predictions,
    labels=["legitimate", "phishing"]
)

print("=" * 50)
print("RAG + Qwen Evaluation - 100 Emails")
print("=" * 50)

print(f"Total emails       : {len(rag_predictions)}")
print(f"Valid predictions  : {len(valid_predictions)}")
print(f"Invalid predictions: {invalid_count}")

print(f"\nAccuracy : {accuracy:.4f} ({accuracy * 100:.2f}%)")
print(f"Precision: {precision:.4f} ({precision * 100:.2f}%)")
print(f"Recall   : {recall:.4f} ({recall * 100:.2f}%)")
print(f"F1 Score : {f1:.4f} ({f1 * 100:.2f}%)")

print("\nConfusion Matrix")
print("                 Predicted")
print("                 Legitimate  Phishing")
print(f"Actual Legitimate    {cm[0][0]:3d}        {cm[0][1]:3d}")
print(f"Actual Phishing      {cm[1][0]:3d}        {cm[1][1]:3d}")

RAG + Qwen Evaluation - 100 Emails
Total emails       : 100
Valid predictions  : 96
Invalid predictions: 4

Accuracy : 0.9896 (98.96%)
Precision: 1.0000 (100.00%)
Recall   : 0.9744 (97.44%)
F1 Score : 0.9870 (98.70%)

Confusion Matrix
                 Predicted
                 Legitimate  Phishing
Actual Legitimate     57          0
Actual Phishing        1         38


# 2977 test emails

In [ ]:
import time
import json
import os

TEST_SIZE = 2977

rag_qwen_predictions = []
rag_qwen_actual_labels = []

start_time = time.time()

for i in range(TEST_SIZE):

    prediction = predict_rag(test_documents[i])

    rag_qwen_predictions.append(prediction)
    rag_qwen_actual_labels.append(test_labels[i])

    print(
        f"{i + 1}/{TEST_SIZE} | "
        f"Actual: {test_labels[i]} | "
        f"Predicted: {prediction}"
    )

    # Save progress every 100 emails
    if (i + 1) % 100 == 0:
        progress_data = {
            "predictions": rag_qwen_predictions,
            "actual_labels": rag_qwen_actual_labels
        }

        progress_path = os.path.join(
            BASE_DIR,
            "results",
            "rag_qwen_2977_progress.json"
        )

        with open(progress_path, "w") as f:
            json.dump(progress_data, f)

        print(f"--- Progress saved: {i + 1}/{TEST_SIZE} ---")

elapsed_time = time.time() - start_time

final_data = {
    "test_size": TEST_SIZE,
    "predictions": rag_qwen_predictions,
    "actual_labels": rag_qwen_actual_labels,
    "time_seconds": elapsed_time
}

final_path = os.path.join(
    BASE_DIR,
    "results",
    "rag_qwen_2977_predictions.json"
)

with open(final_path, "w") as f:
    json.dump(final_data, f)

print("\n" + "=" * 50)
print("RAG + Qwen evaluation completed")
print("Emails tested:", TEST_SIZE)
print("Time taken:", round(elapsed_time, 2), "seconds")
print("Results saved to:", final_path)

1/2977 | Actual: legitimate | Predicted: legitimate
2/2977 | Actual: phishing | Predicted: phishing
3/2977 | Actual: phishing | Predicted: invalid
4/2977 | Actual: phishing | Predicted: phishing
5/2977 | Actual: legitimate | Predicted: legitimate
6/2977 | Actual: phishing | Predicted: phishing
7/2977 | Actual: legitimate | Predicted: legitimate
8/2977 | Actual: legitimate | Predicted: legitimate
9/2977 | Actual: legitimate | Predicted: legitimate
10/2977 | Actual: phishing | Predicted: phishing
11/2977 | Actual: phishing | Predicted: phishing
12/2977 | Actual: legitimate | Predicted: legitimate
13/2977 | Actual: legitimate | Predicted: legitimate
14/2977 | Actual: legitimate | Predicted: legitimate
15/2977 | Actual: legitimate | Predicted: legitimate
16/2977 | Actual: legitimate | Predicted: legitimate
17/2977 | Actual: legitimate | Predicted: legitimate
18/2977 | Actual: legitimate | Predicted: legitimate
19/2977 | Actual: legitimate | Predicted: legitimate
20/2977 | Actual: legitimat

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Keep only valid predictions
valid_indices = [
    i for i, prediction in enumerate(rag_qwen_predictions)
    if prediction in ["legitimate", "phishing"]
]

valid_predictions = [
    rag_qwen_predictions[i]
    for i in valid_indices
]

valid_actual_labels = [
    rag_qwen_actual_labels[i]
    for i in valid_indices
]

invalid_count = len(rag_qwen_predictions) - len(valid_predictions)

accuracy = accuracy_score(
    valid_actual_labels,
    valid_predictions
)

precision = precision_score(
    valid_actual_labels,
    valid_predictions,
    pos_label="phishing",
    zero_division=0
)

recall = recall_score(
    valid_actual_labels,
    valid_predictions,
    pos_label="phishing",
    zero_division=0
)

f1 = f1_score(
    valid_actual_labels,
    valid_predictions,
    pos_label="phishing",
    zero_division=0
)

cm = confusion_matrix(
    valid_actual_labels,
    valid_predictions,
    labels=["legitimate", "phishing"]
)

print("=" * 55)
print("RAG + Qwen FINAL EVALUATION")
print("=" * 55)

print(f"Total emails       : {len(rag_qwen_predictions)}")
print(f"Valid predictions  : {len(valid_predictions)}")
print(f"Invalid predictions: {invalid_count}")

print(f"\nAccuracy : {accuracy:.4f} ({accuracy * 100:.2f}%)")
print(f"Precision: {precision:.4f} ({precision * 100:.2f}%)")
print(f"Recall   : {recall:.4f} ({recall * 100:.2f}%)")
print(f"F1 Score : {f1:.4f} ({f1 * 100:.2f}%)")

print("\nConfusion Matrix")
print("                 Predicted")
print("                 Legitimate  Phishing")
print(f"Actual Legitimate    {cm[0][0]:4d}        {cm[0][1]:4d}")
print(f"Actual Phishing      {cm[1][0]:4d}        {cm[1][1]:4d}")

RAG + Qwen FINAL EVALUATION
Total emails       : 2977
Valid predictions  : 2922
Invalid predictions: 55

Accuracy : 0.9802 (98.02%)
Precision: 0.9715 (97.15%)
Recall   : 0.9870 (98.70%)
F1 Score : 0.9792 (97.92%)

Confusion Matrix
                 Predicted
                 Legitimate  Phishing
Actual Legitimate    1501          40
Actual Phishing        18        1363


## RAG Pipeline Implemented in the Notebook

This RAG system classifies phishing emails by combining retrieval from a vector database with a fine-tuned Qwen language model.

### Pipeline Steps and Corresponding Cells:

1.  **JSONL Data Preparation:**
    *   **Description**: Loading the training and test datasets containing email content and their classification labels (`phishing` or `legitimate`).
    *   **Cells**:
        *   `uKLmqk2kBlSe`: Loads `train.jsonl`.
        *   `FAe-KTvszG6l`: Loads `test.jsonl`.
        *   `9cQtKlsHQW8L`: Reloads `test.jsonl` (for full evaluation).

2.  **Embedding Creation:**
    *   **Description**: Converting email text into numerical vector representations (embeddings) using a pre-trained Sentence Transformer model.
    *   **Cells**:
        *   `YjKtWCsSDHET`: Loads the `SentenceTransformer` model (`all-MiniLM-L6-v2`).
        *   `_rLRP3IIMHS4`: Generates embeddings for the training emails in batches.

3.  **Vector Database Storage (ChromaDB):**
    *   **Description**: Storing the generated embeddings, along with the original email documents and their labels, in a persistent ChromaDB instance.
    *   **Cells**:
        *   `UZkIVCxYBEnR` (and `ZwlOPwLzjGW6` for re-initialization): Initializes the ChromaDB client and gets/creates the `phishing_emails` collection.
        *   `qf5bMSwidtnL`: Adds (upserts) the documents, metadata (labels), and embeddings into the ChromaDB collection.

4.  **Retrieval:**
    *   **Description**: For a given input email, its embedding is generated, and then a similarity search is performed in ChromaDB to retrieve the most semantically similar training emails (and their labels).
    *   **Cells**:
        *   `N0KRSn4ZFLnW`: Demonstrates a test RAG retrieval, showing the input email, query embedding creation, and the top 3 retrieved emails with labels, distances, and content.
        *   `mxyHBdKT51uy`: The `predict_rag` function (steps 1 and 2) performs the query embedding creation and retrieval within the RAG pipeline.

5.  **Relevant Information Formatting for Qwen:**
    *   **Description**: The retrieved similar emails and their labels are formatted into a structured `evidence_text` string, which is then incorporated into the user prompt for the Qwen language model.
    *   **Cells**:
        *   `mxyHBdKT51uy`: The `predict_rag` function (steps 3, 4, and 5) constructs `evidence_text`, defines the `SYSTEM_PROMPT`, and creates the `messages` structure for Qwen.

6.  **Qwen Language Model Generation:**
    *   **Description**: The fine-tuned Qwen model processes the structured prompt (containing the input email and retrieved evidence) to generate a classification.
    *   **Cells**:
        *   `Nbo9sIxih2z6`: Loads the fine-tuned Qwen model and tokenizer (`qwen_phishing_unsloth_correct`).
        *   `mxyHBdKT51uy`: The `predict_rag` function (steps 6, 7, and 8) applies the chat template, tokenizes the input, and generates the model's output.

7.  **Final Answer Extraction:**
    *   **Description**: The raw output from Qwen is decoded, and the classification (`phishing` or `legitimate`) is extracted. If the output format is not as expected, it's marked as `invalid`.
    *   **Cells**:
        *   `mxyHBdKT51uy`: The `predict_rag` function (steps 9 and 10) decodes the generated tokens and extracts the final classification.
        *   `h0J6OKoNjPWH` (and `PjJbTnHJPMVs`): Shows an example of calling `predict_rag` and printing the final classification for a single test email.